# XL-LEXEME Semantic Breadth

This notebook estimates annual semantic breadth for ADHD, Autism, and the three baseline terms. Breadth is operationalised as within-year contextual dispersion among target-aware XL-LEXEME embeddings: higher values indicate that an analysis unit appears in more diverse local contexts during that publication year.


## Setup

The notebook uses the shared LSC mention-context table and publication year (`lsc_year`) as the diachronic axis. The current run uses a fixed cap of 250 contexts per analysis-unit year; set `MAX_CONTEXTS_PER_UNIT_YEAR = None` for an uncapped final-data run. XL-LEXEME is loaded from a local external-resource directory rather than downloaded during analysis.


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
INTERIM_DIR = PROJECT_ROOT / "data/interim/lsc/breadth"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/lsc/breadth"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/breadth"
for directory in [INTERIM_DIR, PROCESSED_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH_CANDIDATES = [
    PROJECT_ROOT / "data/external/models/xl-lexeme",
    PROJECT_ROOT / "data/external/model",
]
XL_LEXEME_MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_UNITS = TARGET_UNITS + BASELINE_UNITS
UNIT_COLOURS = {
    "ADHD": "#0072B2",
    "Autism": "#D55E00",
    "frustration": "#009E73",
    "loneliness": "#CC79A7",
    "sadness": "#6E6E6E",
}
UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}

MAX_CONTEXTS_PER_UNIT_YEAR: int | None = 250
RANDOM_SEED = 123
BOOTSTRAP_REPETITIONS = 500
MIN_CONTEXT_TOKENS = 8
MAX_SEQUENCE_LENGTH = 128
ENCODE_BATCH_SIZE = 8
DEVICE = "cpu"
TARGET_START = "<t>"
TARGET_END = "</t>"
TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
RNG = np.random.default_rng(RANDOM_SEED)


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Shared Contexts

Only WARC-validated, English, deduplicated contexts with parseable publication years enter the shared LSC table. The breadth notebook keeps raw-form, document, and domain metadata so sampling and interpretation remain inspectable.


In [2]:
if not CONTEXT_PATH.exists():
    raise FileNotFoundError(f"Missing shared LSC context table: {CONTEXT_PATH}")
if not XL_LEXEME_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Missing local XL-LEXEME model. Expected one of: "
        + ", ".join(str(path) for path in MODEL_PATH_CANDIDATES)
    )

context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "collapsed_matched_texts",
    "registered_domain",
    "target_sentence",
    "target_sentence_plus_adjacent",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["context_row_id"] = contexts.index.astype(int)
contexts["registered_domain"] = contexts["registered_domain"].fillna("unknown_domain")
contexts["target_group"] = contexts["target_group"].fillna("baseline")

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")

input_summary = pd.DataFrame(
    {
        "metric": ["contexts", "documents", "analysis_units", "years", "model_path"],
        "value": [
            len(contexts),
            contexts["doc_id"].nunique(),
            ", ".join(observed_units),
            f"{min(observed_years)}-{max(observed_years)}",
            str(XL_LEXEME_MODEL_PATH.relative_to(PROJECT_ROOT)),
        ],
    }
)
input_summary


,metric,value
0,contexts,146471
1,documents,106045
2,analysis_units,"ADHD, Autism, frustration, loneliness, sadness"
3,years,2014-2026
4,model_path,data/external/models/xl-lexeme


## Mark Target Contexts

XL-LEXEME needs target-aware contexts. The notebook first tries the target sentence, then falls back to the target sentence plus adjacent context when the sentence is too short or cannot be marked reliably. Rows that cannot be marked are saved as diagnostics and excluded from embedding.


In [3]:
def token_count(text: object) -> int:
    return len(TOKEN_RE.findall(str(text or "")))


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def whitespace_flexible_pattern(text: str) -> str:
    escaped = re.escape(" ".join(str(text).split()))
    return escaped.replace(r"\ ", r"\s+")


def raw_form_patterns(raw_form: str) -> list[str]:
    patterns = {
        "adhd": [r"\bADHD\b"],
        "attention_deficit": [r"\battention\s+deficit(?:\s+hyperactivity(?:\s+disorder)?)?\b"],
        "autism": [r"\bautism\b"],
        "autistic": [r"\bautistic\b"],
        "autism_spectrum": [r"\bautism\s+spectrum\b"],
        "asd_disambiguated": [r"\bASD\b"],
        "frustration": [r"\bfrustration\b"],
        "loneliness": [r"\bloneliness\b"],
        "sadness": [r"\bsadness\b"],
    }
    return patterns.get(raw_form, [r"\b" + re.escape(raw_form.replace("_", " ")) + r"\b"])


def candidate_patterns(row: pd.Series) -> list[tuple[str, str]]:
    candidates: list[tuple[str, str]] = []
    matched_values = [row.get("matched_text")] + split_pipe_values(row.get("collapsed_matched_texts"))
    for value in matched_values:
        if isinstance(value, str) and value.strip():
            candidates.append(("matched_text", whitespace_flexible_pattern(value)))
    for pattern in raw_form_patterns(str(row.get("raw_form") or "")):
        candidates.append(("raw_form", pattern))
    seen: set[str] = set()
    unique_candidates = []
    for source, pattern in candidates:
        if pattern not in seen:
            unique_candidates.append((source, pattern))
            seen.add(pattern)
    return unique_candidates


def mark_first_match(text: str, row: pd.Series) -> tuple[str | None, str | None, str | None]:
    for pattern_source, pattern in candidate_patterns(row):
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            start, end = match.span()
            marked = text[:start] + TARGET_START + text[start:end] + TARGET_END + text[end:]
            return marked, pattern_source, text[start:end]
    return None, None, None


def context_candidates(row: pd.Series) -> list[tuple[str, str]]:
    sentence = str(row.get("target_sentence") or "").strip()
    adjacent = str(row.get("target_sentence_plus_adjacent") or "").strip()
    candidates: list[tuple[str, str]] = []
    if token_count(sentence) >= MIN_CONTEXT_TOKENS:
        candidates.append(("target_sentence", sentence))
        if adjacent and adjacent != sentence:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
    else:
        if adjacent:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
        if sentence:
            candidates.append(("target_sentence", sentence))
    return candidates


def select_marked_context(row: pd.Series) -> dict[str, object]:
    for context_source, text in context_candidates(row):
        marked, pattern_source, marked_text = mark_first_match(text, row)
        if marked:
            return {
                "marked_context": marked,
                "context_source": context_source,
                "mark_pattern_source": pattern_source,
                "marked_text": marked_text,
                "context_token_count": token_count(text),
                "markable": True,
                "unmarkable_reason": "",
            }
    return {
        "marked_context": "",
        "context_source": "",
        "mark_pattern_source": "",
        "marked_text": "",
        "context_token_count": 0,
        "markable": False,
        "unmarkable_reason": "target_span_not_found_in_context",
    }


mark_records = [select_marked_context(row) for _, row in tqdm(contexts.iterrows(), total=len(contexts), desc="Marking target contexts")]
mark_data = pd.DataFrame(mark_records)
marked_contexts = pd.concat([contexts, mark_data], axis=1)

unmarkable_contexts = marked_contexts.loc[~marked_contexts["markable"]].copy()
markable_contexts = marked_contexts.loc[marked_contexts["markable"]].copy()

dedupe_subset = ["analysis_unit", "lsc_year", "doc_id", "marked_context"]
markable_before_dedupe = len(markable_contexts)
markable_contexts = markable_contexts.drop_duplicates(subset=dedupe_subset).reset_index(drop=True)
duplicate_marked_contexts_removed = markable_before_dedupe - len(markable_contexts)

unmarkable_path = INTERIM_DIR / "lsc_breadth_unmarkable_contexts.csv"
unmarkable_columns = [
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "raw_form",
    "matched_text",
    "unmarkable_reason",
]
unmarkable_contexts[unmarkable_columns].to_csv(unmarkable_path, index=False)

pd.DataFrame(
    {
        "metric": ["markable_contexts", "unmarkable_contexts", "duplicate_marked_contexts_removed"],
        "value": [len(markable_contexts), len(unmarkable_contexts), duplicate_marked_contexts_removed],
    }
)


Marking target contexts: 100%|██████████| 146471/146471 [00:05<00:00, 25375.07it/s]


,metric,value
0,markable_contexts,142387
1,unmarkable_contexts,0
2,duplicate_marked_contexts_removed,4084


## Sample Contexts

The breadth calculation can run uncapped, but this execution uses a fixed cap of 250 contexts per analysis-unit year. When a cap applies, sampling is deterministic and stratified by registered domain so one high-volume domain does not dominate a unit-year sample.


In [4]:
def domain_stratified_sample(group: pd.DataFrame, cap: int | None, rng: np.random.Generator) -> pd.DataFrame:
    if cap is None or len(group) <= cap:
        return group.copy()
    domain_counts = group["registered_domain"].value_counts().sort_index()
    domain_names = domain_counts.index.to_numpy()
    ideal = domain_counts.to_numpy(dtype=float) / domain_counts.sum() * cap
    quotas = np.floor(ideal).astype(int)
    remainders = ideal - quotas

    while quotas.sum() < cap:
        candidates = np.where(quotas < domain_counts.to_numpy())[0]
        if len(candidates) == 0:
            break
        best = candidates[np.argmax(remainders[candidates])]
        quotas[best] += 1
        remainders[best] = -1

    sampled_parts = []
    for domain, quota in zip(domain_names, quotas):
        if quota <= 0:
            continue
        domain_rows = group[group["registered_domain"] == domain]
        sampled_index = rng.choice(domain_rows.index.to_numpy(), size=min(quota, len(domain_rows)), replace=False)
        sampled_parts.append(group.loc[sampled_index])

    sampled = pd.concat(sampled_parts, axis=0) if sampled_parts else group.iloc[0:0].copy()
    if len(sampled) < cap:
        remaining = group.drop(index=sampled.index)
        fill_count = min(cap - len(sampled), len(remaining))
        fill_index = rng.choice(remaining.index.to_numpy(), size=fill_count, replace=False)
        sampled = pd.concat([sampled, group.loc[fill_index]], axis=0)
    return sampled.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


available_counts = contexts.groupby(["analysis_unit", "lsc_year"], as_index=False).agg(available_contexts=("doc_id", "size"))
markable_counts = marked_contexts.loc[marked_contexts["markable"]].groupby(["analysis_unit", "lsc_year"], as_index=False).agg(markable_contexts_before_dedupe=("doc_id", "size"))
deduped_counts = markable_contexts.groupby(["analysis_unit", "lsc_year"], as_index=False).agg(markable_contexts=("doc_id", "size"))

sampled_groups = []
for (_, _), group in markable_contexts.groupby(["analysis_unit", "lsc_year"], sort=True):
    sampled_groups.append(domain_stratified_sample(group, MAX_CONTEXTS_PER_UNIT_YEAR, RNG))
sampled_contexts = pd.concat(sampled_groups, axis=0).reset_index(drop=True)
sampled_contexts["embedding_row_id"] = np.arange(len(sampled_contexts), dtype=int)

sampled_counts = (
    sampled_contexts.groupby(["analysis_unit", "lsc_year"], as_index=False)
    .agg(
        sampled_contexts=("doc_id", "size"),
        sampled_documents=("doc_id", "nunique"),
        sampled_domains=("registered_domain", "nunique"),
        target_sentence_contexts=("context_source", lambda values: int((values == "target_sentence").sum())),
        adjacent_contexts=("context_source", lambda values: int((values == "target_sentence_plus_adjacent").sum())),
    )
)

top_domain_share = (
    sampled_contexts.groupby(["analysis_unit", "lsc_year", "registered_domain"], as_index=False)
    .size()
    .rename(columns={"size": "domain_contexts"})
)
top_domain_share["sampled_contexts_for_share"] = top_domain_share.groupby(["analysis_unit", "lsc_year"])["domain_contexts"].transform("sum")
top_domain_share["domain_share"] = top_domain_share["domain_contexts"] / top_domain_share["sampled_contexts_for_share"]
top_domain_share = top_domain_share.sort_values("domain_share", ascending=False).groupby(["analysis_unit", "lsc_year"], as_index=False).head(1)
top_domain_share = top_domain_share.rename(columns={"registered_domain": "top_domain", "domain_share": "top_domain_share"})[
    ["analysis_unit", "lsc_year", "top_domain", "top_domain_share"]
]

sampling_diagnostics = available_counts.merge(markable_counts, on=["analysis_unit", "lsc_year"], how="left")
sampling_diagnostics = sampling_diagnostics.merge(deduped_counts, on=["analysis_unit", "lsc_year"], how="left")
sampling_diagnostics = sampling_diagnostics.merge(sampled_counts, on=["analysis_unit", "lsc_year"], how="left")
sampling_diagnostics = sampling_diagnostics.merge(top_domain_share, on=["analysis_unit", "lsc_year"], how="left")
for column in ["markable_contexts_before_dedupe", "markable_contexts", "sampled_contexts", "sampled_documents", "sampled_domains"]:
    sampling_diagnostics[column] = sampling_diagnostics[column].fillna(0).astype(int)
sampling_diagnostics["unmarkable_contexts"] = sampling_diagnostics["available_contexts"] - sampling_diagnostics["markable_contexts_before_dedupe"]
sampling_diagnostics["duplicate_marked_contexts_removed"] = sampling_diagnostics["markable_contexts_before_dedupe"] - sampling_diagnostics["markable_contexts"]
sampling_diagnostics["unmarkable_share"] = sampling_diagnostics["unmarkable_contexts"] / sampling_diagnostics["available_contexts"].replace(0, np.nan)
sampling_diagnostics["cap_applied"] = sampling_diagnostics["sampled_contexts"] < sampling_diagnostics["markable_contexts"]
sampling_diagnostics["max_contexts_per_unit_year"] = -1 if MAX_CONTEXTS_PER_UNIT_YEAR is None else MAX_CONTEXTS_PER_UNIT_YEAR
sampling_diagnostics = sampling_diagnostics.sort_values(["analysis_unit", "lsc_year"]).reset_index(drop=True)

raw_form_diagnostics = (
    sampled_contexts.groupby(["analysis_unit", "lsc_year", "term_role", "target_group", "raw_form"], as_index=False)
    .agg(sampled_contexts=("doc_id", "size"), sampled_documents=("doc_id", "nunique"))
)
raw_form_diagnostics["raw_form_context_share"] = raw_form_diagnostics["sampled_contexts"] / raw_form_diagnostics.groupby(["analysis_unit", "lsc_year"])["sampled_contexts"].transform("sum")

sampled_contexts_path = INTERIM_DIR / "lsc_breadth_sampled_contexts.parquet"
sampling_diagnostics_path = PROCESSED_DIR / "lsc_breadth_sampling_diagnostics.csv"
raw_form_diagnostics_path = PROCESSED_DIR / "lsc_breadth_raw_form_diagnostics.csv"
sampled_contexts.to_parquet(sampled_contexts_path, index=False)
sampling_diagnostics.to_csv(sampling_diagnostics_path, index=False)
raw_form_diagnostics.to_csv(raw_form_diagnostics_path, index=False)

sampling_diagnostics.head(10)


,analysis_unit,lsc_year,available_contexts,markable_contexts_before_dedupe,markable_contexts,sampled_contexts,sampled_documents,sampled_domains,target_sentence_contexts,adjacent_contexts,top_domain,top_domain_share,unmarkable_contexts,duplicate_marked_contexts_removed,unmarkable_share,cap_applied,max_contexts_per_unit_year
0,ADHD,2014,1587,1587,1511,250,244,215,238,12,naturalnews.com,0.084,0,76,0.0,True,250
1,ADHD,2015,1176,1176,1111,250,247,228,243,7,rightdiagnosis.com,0.024,0,65,0.0,True,250
2,ADHD,2016,1336,1336,1276,250,248,239,237,13,apples4theteacher.com,0.028,0,60,0.0,True,250
3,ADHD,2017,1297,1297,1236,250,248,232,237,13,psychologytoday.com,0.020,0,61,0.0,True,250
4,ADHD,2018,1331,1331,1267,250,250,244,238,12,emedtv.com,0.012,0,64,0.0,True,250
5,ADHD,2019,1138,1138,1080,250,248,244,242,8,healthyplace.com,0.008,0,58,0.0,True,250
6,ADHD,2020,1132,1132,1077,250,250,244,238,12,apples4theteacher.com,0.012,0,55,0.0,True,250
7,ADHD,2021,1146,1146,1097,250,247,239,244,6,naturalnews.com,0.012,0,49,0.0,True,250
8,ADHD,2022,1186,1186,1109,250,248,245,241,9,healthyplace.com,0.012,0,77,0.0,True,250
9,ADHD,2023,950,950,898,250,248,240,245,5,psychologytoday.com,0.016,0,52,0.0,True,250


## Encode XL-LEXEME Contexts

The local XL-LEXEME model is used through `torch` and `transformers`. Contexts are tokenised with target markers, truncated around the marked target when necessary, and represented by the mean hidden state of the target tokens between `<t>` and `</t>`.


In [5]:
try:
    import torch
    from transformers import AutoModel, AutoTokenizer
except ImportError as exc:
    raise ImportError(
        "Semantic breadth requires torch and transformers. Install/update the msc-nlp environment from environment.yml."
    ) from exc


torch.set_num_threads(2)
tokenizer = AutoTokenizer.from_pretrained(str(XL_LEXEME_MODEL_PATH), local_files_only=True)
# Long contexts are manually target-centred before model inference.
tokenizer.model_max_length = 100_000
model = AutoModel.from_pretrained(str(XL_LEXEME_MODEL_PATH), local_files_only=True).to(DEVICE)
model.eval()

start_marker_id = tokenizer.convert_tokens_to_ids(TARGET_START)
end_marker_id = tokenizer.convert_tokens_to_ids(TARGET_END)
if start_marker_id == tokenizer.unk_token_id or end_marker_id == tokenizer.unk_token_id:
    raise RuntimeError("XL-LEXEME tokenizer does not recognise target marker tokens <t> and </t>.")


def find_marker_pair(token_ids: list[int]) -> tuple[int, int]:
    try:
        start = token_ids.index(start_marker_id)
        end = token_ids.index(end_marker_id, start + 1)
    except ValueError as exc:
        raise RuntimeError("Tokenised context is missing XL-LEXEME target markers.") from exc
    if end <= start + 1:
        raise RuntimeError("XL-LEXEME target markers contain no target tokens.")
    return start, end


def build_model_inputs(marked_context: str) -> tuple[list[int], list[int], bool]:
    token_ids = tokenizer.encode(marked_context, add_special_tokens=False)
    start, end = find_marker_pair(token_ids)
    payload_max = MAX_SEQUENCE_LENGTH - 2
    truncated = len(token_ids) > payload_max
    if truncated:
        target_center = (start + end) // 2
        window_start = max(0, target_center - payload_max // 2)
        window_start = min(window_start, max(0, len(token_ids) - payload_max))
        if start < window_start:
            window_start = max(0, start - 1)
        if end >= window_start + payload_max:
            window_start = max(0, end - payload_max + 1)
        window_end = min(len(token_ids), window_start + payload_max)
        token_ids = token_ids[window_start:window_end]
        start, end = find_marker_pair(token_ids)

    input_ids = [tokenizer.cls_token_id] + token_ids + [tokenizer.sep_token_id]
    target_mask = [0] * len(input_ids)
    for position in range(start + 2, end + 1):  # +1 for CLS, exclude start marker, exclude end marker.
        target_mask[position] = 1
    return input_ids, target_mask, truncated


def encode_marked_contexts(marked_texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
    embeddings = []
    truncated_flags = []
    for start in tqdm(range(0, len(marked_texts), ENCODE_BATCH_SIZE), desc="Encoding XL-LEXEME contexts"):
        batch_texts = marked_texts[start : start + ENCODE_BATCH_SIZE]
        built = [build_model_inputs(text) for text in batch_texts]
        batch_input_ids, batch_target_masks, batch_truncated = zip(*built)
        max_len = max(len(ids) for ids in batch_input_ids)
        padded_ids = []
        attention_masks = []
        padded_target_masks = []
        for input_ids, target_mask in zip(batch_input_ids, batch_target_masks):
            pad_len = max_len - len(input_ids)
            padded_ids.append(input_ids + [tokenizer.pad_token_id] * pad_len)
            attention_masks.append([1] * len(input_ids) + [0] * pad_len)
            padded_target_masks.append(target_mask + [0] * pad_len)
        input_tensor = torch.tensor(padded_ids, dtype=torch.long, device=DEVICE)
        attention_tensor = torch.tensor(attention_masks, dtype=torch.long, device=DEVICE)
        target_mask_tensor = torch.tensor(padded_target_masks, dtype=torch.bool, device=DEVICE)
        with torch.no_grad():
            hidden = model(input_ids=input_tensor, attention_mask=attention_tensor).last_hidden_state
        for row_index in range(hidden.shape[0]):
            target_hidden = hidden[row_index][target_mask_tensor[row_index]]
            embeddings.append(target_hidden.mean(dim=0).cpu().numpy().astype("float32"))
        truncated_flags.extend(batch_truncated)
    return np.vstack(embeddings).astype("float32"), np.array(truncated_flags, dtype=bool)


embeddings, truncated_for_model = encode_marked_contexts(sampled_contexts["marked_context"].tolist())
embedding_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
if np.any(embedding_norms == 0):
    raise RuntimeError("One or more XL-LEXEME embeddings has zero norm.")
embeddings_normalised = embeddings / embedding_norms

sampled_contexts["truncated_for_model"] = truncated_for_model
embedding_index = sampled_contexts[
    [
        "embedding_row_id",
        "context_row_id",
        "doc_id",
        "lsc_year",
        "analysis_unit",
        "term_role",
        "target_group",
        "raw_form",
        "registered_domain",
        "context_source",
        "context_token_count",
        "truncated_for_model",
    ]
].copy()

embedding_path = INTERIM_DIR / "lsc_breadth_embeddings.npy"
embedding_normalised_path = INTERIM_DIR / "lsc_breadth_embeddings_normalised.npy"
embedding_index_path = INTERIM_DIR / "lsc_breadth_embedding_index.csv"
np.save(embedding_path, embeddings)
np.save(embedding_normalised_path, embeddings_normalised)
embedding_index.to_csv(embedding_index_path, index=False)
sampled_contexts.to_parquet(sampled_contexts_path, index=False)

pd.DataFrame(
    {
        "metric": ["sampled_contexts", "embedding_dimensions", "contexts_truncated_for_model"],
        "value": [len(sampled_contexts), embeddings.shape[1], int(truncated_for_model.sum())],
    }
)


Encoding XL-LEXEME contexts: 100%|██████████| 2032/2032 [1:50:18<00:00,  3.26s/it]  


,metric,value
0,sampled_contexts,16250
1,embedding_dimensions,1024
2,contexts_truncated_for_model,805


## Annual Breadth Scores

Breadth is the mean pairwise cosine distance within each analysis-unit year. The calculation uses L2-normalised embeddings and a closed-form expression, avoiding materialising all pairwise distances for the main score.


In [6]:
def mean_pairwise_cosine_distance(normalised_vectors: np.ndarray) -> float:
    n = normalised_vectors.shape[0]
    if n < 2:
        return float("nan")
    sum_vector = normalised_vectors.sum(axis=0)
    sum_pairwise_similarity = (float(np.dot(sum_vector, sum_vector)) - n) / 2.0
    pair_count = n * (n - 1) / 2.0
    mean_similarity = sum_pairwise_similarity / pair_count
    return float(1.0 - mean_similarity)


def bootstrap_document_breadth(group_index: pd.DataFrame, normalised_vectors: np.ndarray, rng: np.random.Generator) -> dict[str, object]:
    document_to_rows = [values.to_numpy(dtype=int) for _, values in group_index.groupby("doc_id")["embedding_row_id"]]
    if len(document_to_rows) < 2:
        return {
            "bootstrap_repetitions": 0,
            "bootstrap_unit": "doc_id",
            "breadth_bootstrap_mean": float("nan"),
            "breadth_ci_low": float("nan"),
            "breadth_ci_high": float("nan"),
        }
    bootstrap_values = []
    for _ in range(BOOTSTRAP_REPETITIONS):
        sampled_docs = rng.integers(0, len(document_to_rows), len(document_to_rows))
        sampled_rows = np.concatenate([document_to_rows[index] for index in sampled_docs])
        if len(sampled_rows) >= 2:
            bootstrap_values.append(mean_pairwise_cosine_distance(normalised_vectors[sampled_rows]))
    return {
        "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "bootstrap_unit": "doc_id",
        "breadth_bootstrap_mean": float(np.mean(bootstrap_values)),
        "breadth_ci_low": float(np.quantile(bootstrap_values, 0.025)),
        "breadth_ci_high": float(np.quantile(bootstrap_values, 0.975)),
    }


breadth_records = []
for (unit, year), group in embedding_index.groupby(["analysis_unit", "lsc_year"], sort=True):
    row_ids = group["embedding_row_id"].to_numpy(dtype=int)
    vectors = embeddings_normalised[row_ids]
    metadata = group.iloc[0]
    bootstrap = bootstrap_document_breadth(group, embeddings_normalised, RNG)
    breadth_records.append(
        {
            "lsc_year": int(year),
            "analysis_unit": unit,
            "term_role": metadata["term_role"],
            "target_group": metadata["target_group"],
            "breadth_mean_pairwise_cosine_distance": mean_pairwise_cosine_distance(vectors),
            "sampled_contexts": int(len(group)),
            "sampled_documents": int(group["doc_id"].nunique()),
            "sampled_domains": int(group["registered_domain"].nunique()),
            "contexts_truncated_for_model": int(group["truncated_for_model"].sum()),
            **bootstrap,
        }
    )

annual_breadth = pd.DataFrame(breadth_records).sort_values(["analysis_unit", "lsc_year"]).reset_index(drop=True)
annual_breadth = annual_breadth.merge(
    sampling_diagnostics[
        [
            "analysis_unit",
            "lsc_year",
            "available_contexts",
            "markable_contexts",
            "unmarkable_contexts",
            "unmarkable_share",
            "cap_applied",
            "top_domain",
            "top_domain_share",
            "max_contexts_per_unit_year",
        ]
    ],
    on=["analysis_unit", "lsc_year"],
    how="left",
)

flag_rows = []
for row in annual_breadth.itertuples(index=False):
    flags = []
    if row.sampled_contexts < 100:
        flags.append("sampled_contexts_lt_100")
    if row.sampled_documents < 50:
        flags.append("sampled_documents_lt_50")
    if row.unmarkable_share > 0.05:
        flags.append("unmarkable_share_gt_0_05")
    if pd.notna(row.top_domain_share) and row.top_domain_share > 0.20:
        flags.append("top_domain_share_gt_0_20")
    if row.contexts_truncated_for_model / max(row.sampled_contexts, 1) > 0.50:
        flags.append("truncated_context_share_gt_0_50")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "flags": ";".join(flags),
                "sampled_contexts": row.sampled_contexts,
                "sampled_documents": row.sampled_documents,
                "unmarkable_share": row.unmarkable_share,
                "top_domain_share": row.top_domain_share,
                "contexts_truncated_for_model": row.contexts_truncated_for_model,
            }
        )

audit_flag_columns = [
    "lsc_year",
    "analysis_unit",
    "flags",
    "sampled_contexts",
    "sampled_documents",
    "unmarkable_share",
    "top_domain_share",
    "contexts_truncated_for_model",
]
audit_flags = pd.DataFrame(flag_rows, columns=audit_flag_columns)

annual_breadth_path = PROCESSED_DIR / "lsc_breadth_annual_scores.csv"
audit_flags_path = PROCESSED_DIR / "lsc_breadth_audit_flags.csv"
annual_breadth.to_csv(annual_breadth_path, index=False)
audit_flags.to_csv(audit_flags_path, index=False)

annual_breadth.head(10)


,lsc_year,analysis_unit,term_role,target_group,breadth_mean_pairwise_cosine_distance,sampled_contexts,sampled_documents,sampled_domains,contexts_truncated_for_model,bootstrap_repetitions,bootstrap_unit,breadth_bootstrap_mean,breadth_ci_low,breadth_ci_high,available_contexts,markable_contexts,unmarkable_contexts,unmarkable_share,cap_applied,top_domain,top_domain_share,max_contexts_per_unit_year
0,2014,ADHD,target,ADHD,0.134988,250,244,215,28,500,doc_id,0.134099,0.119178,0.147552,1587,1511,0,0.0,True,naturalnews.com,0.084,250
1,2015,ADHD,target,ADHD,0.145252,250,247,228,22,500,doc_id,0.144693,0.127109,0.160582,1176,1111,0,0.0,True,rightdiagnosis.com,0.024,250
2,2016,ADHD,target,ADHD,0.152279,250,248,239,15,500,doc_id,0.151226,0.134037,0.169012,1336,1276,0,0.0,True,apples4theteacher.com,0.028,250
3,2017,ADHD,target,ADHD,0.139178,250,248,232,16,500,doc_id,0.138846,0.123330,0.155464,1297,1236,0,0.0,True,psychologytoday.com,0.020,250
4,2018,ADHD,target,ADHD,0.136370,250,250,244,15,500,doc_id,0.135425,0.116712,0.151916,1331,1267,0,0.0,True,emedtv.com,0.012,250
5,2019,ADHD,target,ADHD,0.134676,250,248,244,14,500,doc_id,0.134227,0.116098,0.153444,1138,1080,0,0.0,True,healthyplace.com,0.008,250
6,2020,ADHD,target,ADHD,0.129813,250,250,244,18,500,doc_id,0.128864,0.112339,0.143566,1132,1077,0,0.0,True,apples4theteacher.com,0.012,250
7,2021,ADHD,target,ADHD,0.141037,250,247,239,22,500,doc_id,0.140833,0.124732,0.155839,1146,1097,0,0.0,True,naturalnews.com,0.012,250
8,2022,ADHD,target,ADHD,0.131556,250,248,245,15,500,doc_id,0.131276,0.115446,0.148446,1186,1109,0,0.0,True,healthyplace.com,0.012,250
9,2023,ADHD,target,ADHD,0.133768,250,248,240,17,500,doc_id,0.133159,0.117293,0.149562,950,898,0,0.0,True,psychologytoday.com,0.016,250


## Breadth Trajectories

Targets and baselines are plotted separately. The ribbons show document-level bootstrap confidence intervals around each annual contextual-dispersion estimate.


In [7]:
def style_axis(ax: plt.Axes) -> None:
    ax.set_xlim(min(EXPECTED_YEARS) - 0.25, max(EXPECTED_YEARS) + 0.25)
    ax.set_xticks(EXPECTED_YEARS)
    ax.tick_params(axis="x", rotation=45)
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Mean pairwise cosine distance")


fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), sharey=True, constrained_layout=True)
for ax, units, title in zip(axes, [TARGET_UNITS, BASELINE_UNITS], ["Target groups", "Baseline terms"]):
    for unit in units:
        unit_data = annual_breadth[annual_breadth["analysis_unit"] == unit].sort_values("lsc_year")
        years = unit_data["lsc_year"].to_numpy(dtype=float)
        mean = unit_data["breadth_mean_pairwise_cosine_distance"].to_numpy(dtype=float)
        ci_low = unit_data["breadth_ci_low"].to_numpy(dtype=float)
        ci_high = unit_data["breadth_ci_high"].to_numpy(dtype=float)
        ax.plot(
            years,
            mean,
            label=unit,
            color=UNIT_COLOURS[unit],
            marker=UNIT_MARKERS[unit],
            linewidth=2.2,
            markersize=5.5,
        )
        ax.fill_between(years, ci_low, ci_high, color=UNIT_COLOURS[unit], alpha=0.15, linewidth=0)
    ax.set_title(title)
    style_axis(ax)
    ax.legend(loc="best")

fig.suptitle("Annual semantic breadth from XL-LEXEME contexts", fontsize=13, y=1.03)
trajectory_png = FIGURE_DIR / "lsc_breadth_trajectories.png"
trajectory_pdf = FIGURE_DIR / "lsc_breadth_trajectories.pdf"
fig.savefig(trajectory_png, dpi=300, bbox_inches="tight")
fig.savefig(trajectory_pdf, bbox_inches="tight")
plt.close(fig)


## Handoff Summary

The main handoff for downstream synthesis is the annual breadth score table. The sampled contexts, embedding arrays, embedding index, sampling diagnostics, raw-form diagnostics, and audit flags make the embedding run reproducible and inspectable.


In [8]:
expected_annual_rows = len(EXPECTED_YEARS) * len(EXPECTED_UNITS)
summary = {
    "annual_rows": len(annual_breadth),
    "expected_annual_rows": expected_annual_rows,
    "sampled_context_rows": len(sampled_contexts),
    "embedding_rows": int(embeddings.shape[0]),
    "embedding_dimensions": int(embeddings.shape[1]),
    "audit_flag_rows": len(audit_flags),
    "max_sampled_contexts_per_unit_year": int(sampling_diagnostics["sampled_contexts"].max()),
    "contexts_truncated_for_model": int(embedding_index["truncated_for_model"].sum()),
}
if summary["annual_rows"] != expected_annual_rows:
    raise RuntimeError(f"Expected {expected_annual_rows} annual rows, found {summary['annual_rows']}.")
if summary["embedding_rows"] != len(embedding_index):
    raise RuntimeError("Embedding row count does not match embedding index row count.")
if MAX_CONTEXTS_PER_UNIT_YEAR is not None and summary["max_sampled_contexts_per_unit_year"] > MAX_CONTEXTS_PER_UNIT_YEAR:
    raise RuntimeError("Sampling cap was exceeded.")
summary


{'annual_rows': 65,
 'expected_annual_rows': 65,
 'sampled_context_rows': 16250,
 'embedding_rows': 16250,
 'embedding_dimensions': 1024,
 'audit_flag_rows': 0,
 'max_sampled_contexts_per_unit_year': 250,
 'contexts_truncated_for_model': 805}